# Module 04 — Notebook 01: Occlusion on Image Classification

## Learning Objectives
By completing this notebook, you will:
1. Apply `Occlusion` from Captum to produce heatmaps for ResNet-50
2. Compare Occlusion at different window sizes and strides
3. Visualize positive (helpful) and negative (confusing) attribution regions
4. Compare Occlusion vs. Integrated Gradients on the same images

## Prerequisites
- Module 02 (Integrated Gradients)
- Guide 01 (Occlusion theory)

## Estimated Time: 14 minutes

---

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import io
import json
from PIL import Image

from captum.attr import Occlusion, IntegratedGradients

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(t.cpu() * std + mean, 0, 1)

def norm99(arr):
    """Percentile normalization to [0,1] using 1st/99th percentiles."""
    p1  = np.percentile(arr, 1)
    p99 = np.percentile(arr, 99)
    return np.clip((arr - p1) / (p99 - p1 + 1e-8), 0, 1)

model = models.resnet50(weights='IMAGENET1K_V1').to(DEVICE)
model.eval()
print("ResNet-50 loaded.")

In [ ]:
# Load ImageNet labels and dog image
LABELS_URL = (
    "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels"
    "/master/imagenet-simple-labels.json"
)
try:
    with urllib.request.urlopen(LABELS_URL) as resp:
        labels = json.loads(resp.read().decode())
except Exception:
    labels = [f"class_{i}" for i in range(1000)]
    labels[208] = "Labrador retriever"

DOG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg"
try:
    with urllib.request.urlopen(DOG_URL, timeout=10) as resp:
        raw = Image.open(io.BytesIO(resp.read())).convert('RGB')
except Exception:
    raw = Image.fromarray(np.random.randint(80, 200, (320, 320, 3), dtype=np.uint8))

input_tensor = preprocess(raw).unsqueeze(0).to(DEVICE)
img_np = denormalize(input_tensor.squeeze(0)).permute(1, 2, 0).numpy()
baseline = torch.zeros_like(input_tensor)

with torch.no_grad():
    logits = model(input_tensor)
    top_class = logits.argmax().item()

print(f"Prediction: {labels[top_class]} (class {top_class})")

## 1. Occlusion at Different Resolutions

We compare three window/stride combinations to show the resolution vs. computation trade-off:
- **Coarse** (window=30, stride=15): ~225 passes, low resolution
- **Medium** (window=15, stride=8): ~784 passes, good balance
- **Fine** (window=7, stride=4): ~3,136 passes, high resolution

Note: smaller strides and windows are more expensive. Captum batches the forward passes efficiently.

In [ ]:
occ = Occlusion(model)

# Three resolution configs: (window_size, stride)
configs = [
    (30, 15, 'Coarse (~225 passes)'),
    (15, 8,  'Medium (~784 passes)'),
    (7,  4,  'Fine (~3136 passes)'),
]

occ_maps = []
for window_size, stride, label in configs:
    print(f"Computing Occlusion: window={window_size}×{window_size}, stride={stride}...")
    attr = occ.attribute(
        input_tensor,
        strides=(3, stride, stride),                        # Stride all channels together
        target=top_class,
        sliding_window_shapes=(3, window_size, window_size), # Window covers all channels
        baselines=0                                          # Black baseline
    )
    # Aggregate across color channels
    heatmap = attr.abs().mean(dim=1).squeeze(0).detach().cpu().numpy()
    occ_maps.append((norm99(heatmap), label))

print("All occlusion maps computed.")

In [ ]:
# Visualize: resolution comparison
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle(
    f'Occlusion Resolution Comparison — {labels[top_class]}',
    fontsize=13
)

# Column 0: original image
for row in range(2):
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title('Original', fontsize=10)
    axes[row, 0].axis('off')

# Columns 1–3: each resolution
for col, (heatmap, lbl) in enumerate(occ_maps, start=1):
    # Pure heatmap
    im = axes[0, col].imshow(heatmap, cmap='hot', vmin=0, vmax=1)
    axes[0, col].set_title(f'Occlusion\n{lbl}', fontsize=9)
    axes[0, col].axis('off')
    plt.colorbar(im, ax=axes[0, col], fraction=0.046)

    # Overlay
    axes[1, col].imshow(img_np)
    axes[1, col].imshow(heatmap, alpha=0.55, cmap='hot', vmin=0, vmax=1)
    axes[1, col].set_title(f'Overlay\n{lbl}', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

print("Observation: finer windows produce crisper heatmaps but at higher computational cost.")

## 2. Positive vs. Negative Attribution

Occlusion can produce **negative attribution**: regions where occluding *increases* the model's confidence. This reveals areas that are actually confusing the model — potential spurious correlations.

We use the signed attribution (not abs) to see both.

In [ ]:
# Compute signed occlusion attribution (medium resolution)
attr_signed = occ.attribute(
    input_tensor,
    strides=(3, 8, 8),
    target=top_class,
    sliding_window_shapes=(3, 15, 15),
    baselines=0
)

# Separate positive and negative contributions
attr_2d_signed = attr_signed.mean(dim=1).squeeze(0).detach().cpu().numpy()

positive_map = np.clip(attr_2d_signed, 0, None)   # Regions that help prediction
negative_map = np.clip(-attr_2d_signed, 0, None)  # Regions that confuse prediction

# Normalize each map independently
pos_norm = positive_map / (positive_map.max() + 1e-8)
neg_norm = negative_map / (negative_map.max() + 1e-8)

print(f"Positive attribution: {(attr_2d_signed > 0).mean():.1%} of pixels")
print(f"Negative attribution: {(attr_2d_signed < 0).mean():.1%} of pixels")
print(f"Max positive attribution: {positive_map.max():.5f}")
print(f"Max negative attribution: {negative_map.max():.5f}")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle(
    f'Signed Occlusion: Positive (helpful) vs Negative (confusing) — {labels[top_class]}',
    fontsize=12
)

axes[0].imshow(img_np)
axes[0].set_title('Input Image')
axes[0].axis('off')

# Signed heatmap using diverging colormap
max_abs = np.abs(attr_2d_signed).max()
im1 = axes[1].imshow(attr_2d_signed, cmap='RdBu_r',
                      vmin=-max_abs, vmax=max_abs)
axes[1].set_title('Signed Attribution\n(blue=negative, red=positive)')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# Positive map
axes[2].imshow(img_np)
axes[2].imshow(pos_norm, alpha=0.6, cmap='Greens', vmin=0, vmax=1)
axes[2].set_title('Positive Attribution\n(occluding this HURTS prediction)')
axes[2].axis('off')

# Negative map
axes[3].imshow(img_np)
axes[3].imshow(neg_norm, alpha=0.6, cmap='Reds', vmin=0, vmax=1)
axes[3].set_title('Negative Attribution\n(occluding this HELPS prediction)')
axes[3].axis('off')

plt.tight_layout()
plt.show()

## 3. Occlusion vs. Integrated Gradients

Both methods should identify similar important regions. Systematic disagreement indicates either:
- Gradient saturation in IG (rare with proper integration)
- Feature interaction effects captured by Occlusion but not IG
- Noise in the Occlusion map (too coarse a window)

We compute both and measure their spatial rank correlation.

In [ ]:
from scipy.stats import spearmanr

# Compute IG
print("Computing Integrated Gradients...")
ig = IntegratedGradients(model)
inp = input_tensor.clone().requires_grad_(True)
ig_attr, ig_delta = ig.attribute(
    inp, baselines=baseline, target=top_class,
    n_steps=50, return_convergence_delta=True
)
print(f"IG convergence delta: {ig_delta.item():.5f}")

ig_map = norm99(
    ig_attr.abs().mean(dim=1).squeeze(0).detach().cpu().numpy()
)

# Use medium-resolution Occlusion
occ_map = occ_maps[1][0]  # Medium resolution

# Compute Spearman rank correlation
# Flatten to 1D for correlation
rho, p_val = spearmanr(ig_map.flatten(), occ_map.flatten())
print(f"Spearman rank correlation (IG vs Occlusion): {rho:.4f} (p={p_val:.2e})")

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle(
    f'Occlusion vs IG — {labels[top_class]} — Spearman ρ = {rho:.3f}',
    fontsize=12
)

axes[0].imshow(img_np)
axes[0].set_title('Input Image')
axes[0].axis('off')

for ax, hmap, title in [
    (axes[1], ig_map, f'Integrated Gradients\n(delta={ig_delta.item():.4f})'),
    (axes[2], occ_map, 'Occlusion (window=15, stride=8)'),
]:
    ax.imshow(img_np)
    ax.imshow(hmap, alpha=0.55, cmap='hot', vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

# Difference map
diff = np.abs(ig_map - occ_map)
im4 = axes[3].imshow(diff, cmap='viridis', vmin=0, vmax=1)
axes[3].set_title('|IG - Occlusion|\n(disagreement map)', fontsize=10)
axes[3].axis('off')
plt.colorbar(im4, ax=axes[3], fraction=0.046)

plt.tight_layout()
plt.show()

print(f"\nInterpretation:")
if rho > 0.6:
    print(f"  High agreement (ρ={rho:.3f}): both methods identify similar important regions.")
elif rho > 0.3:
    print(f"  Moderate agreement (ρ={rho:.3f}): generally consistent with some local differences.")
else:
    print(f"  Low agreement (ρ={rho:.3f}): methods disagree — investigate further.")

## 4. Exercise: Baseline Sensitivity of Occlusion

The choice of baseline value in Occlusion (what to fill the window with) affects the result. Compare three baselines:
- `baselines=0` (black pixel)
- `baselines=0.5` (grey pixel, mid-range after normalization)
- `baselines=img_mean` (mean pixel value of the input image)

**Question:** Which regions are robust to baseline choice?

In [ ]:
# Compute Occlusion with three different baselines

# Mean pixel of the input image (in normalized space)
img_mean_val = input_tensor.mean().item()

baseline_configs = [
    (0,             'Zero baseline (black)'),
    (img_mean_val,  f'Image mean baseline ({img_mean_val:.3f})'),
    (0.5,           'Mid-range baseline (0.5)'),
]

baseline_maps = []
print("Computing Occlusion with 3 different baselines...")
for bl_val, bl_name in baseline_configs:
    attr = occ.attribute(
        input_tensor,
        strides=(3, 8, 8),
        target=top_class,
        sliding_window_shapes=(3, 15, 15),
        baselines=bl_val
    )
    hm = norm99(attr.abs().mean(1).squeeze(0).detach().cpu().numpy())
    baseline_maps.append((hm, bl_name))
    print(f"  Done: {bl_name}")

# Compute pairwise Spearman correlations
print("\nPairwise Spearman correlations:")
for i in range(len(baseline_maps)):
    for j in range(i+1, len(baseline_maps)):
        rho, _ = spearmanr(
            baseline_maps[i][0].flatten(),
            baseline_maps[j][0].flatten()
        )
        print(f"  {baseline_maps[i][1][:20]:20s} vs {baseline_maps[j][1][:20]:20s}: ρ={rho:.4f}")

# Consensus map: regions important across all baselines
stacked = np.stack([m for m, _ in baseline_maps])
consensus = stacked.min(axis=0)  # Conservative: minimum across baselines

# Visualize
fig, axes = plt.subplots(1, 5, figsize=(24, 5))
fig.suptitle('Occlusion Baseline Sensitivity (window=15, stride=8)', fontsize=12)

axes[0].imshow(img_np)
axes[0].set_title('Input')
axes[0].axis('off')

for ax, (hm, name) in zip(axes[1:4], baseline_maps):
    ax.imshow(img_np)
    ax.imshow(hm, alpha=0.55, cmap='hot', vmin=0, vmax=1)
    ax.set_title(name[:25], fontsize=9)
    ax.axis('off')

axes[4].imshow(img_np)
axes[4].imshow(consensus / (consensus.max() + 1e-8),
                alpha=0.55, cmap='hot', vmin=0, vmax=1)
axes[4].set_title('Consensus\n(min across baselines)', fontsize=9)
axes[4].axis('off')

plt.tight_layout()
plt.show()

# Auto-check: all maps should be non-negative after norm99
for hm, name in baseline_maps:
    assert hm.min() >= 0.0, f"Heatmap should be non-negative: {name}"
    assert hm.max() <= 1.0, f"Heatmap should be <= 1.0: {name}"
print("\nAll checks passed.")

## Summary

### What You Learned

1. **Occlusion** systematically blocks image regions and measures the resulting drop in model confidence
2. **Window/stride trade-off:** smaller windows and strides produce finer heatmaps at higher computational cost
3. **Positive attribution:** occluding this region hurts the prediction (the region is helpful)
4. **Negative attribution:** occluding this region helps the prediction (the region is confusing the model)
5. **Occlusion vs. IG:** high Spearman correlation (ρ > 0.6) indicates converging evidence from both methods
6. **Baseline sensitivity:** regions appearing in the consensus map (across multiple baselines) are the most robust

### Key API
```python
from captum.attr import Occlusion

occ = Occlusion(model)
attr = occ.attribute(
    input_tensor,
    strides=(3, 8, 8),
    target=class_idx,
    sliding_window_shapes=(3, 15, 15),
    baselines=0
)
```

### What's Next

**Notebook 02:** Feature Ablation on tabular data — apply perturbation attribution to wine quality prediction, where the features are chemical properties rather than pixels.